In [3]:
from pathlib import Path

# Step 1: Check if your file is uploaded
input_file = Path("/home/lyz/repos/open-r1-lean/past_runs/run_3_algebra/output.log")


In [5]:
# Just peek at the beginning to confirm structure
with open(input_file, 'r') as f:
    lines = f.readlines()[:100]
for i, line in enumerate(lines):
    print(f"{i:3d}: {repr(line)}")  # repr shows hidden chars

  0: 'wandb: Detected [huggingface_hub.inference, openai] in use.\n'
  1: 'wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.\n'
  2: 'wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/\n'
  3: '  0%|          | 0/20 [00:00<?, ?it/s]\n'
  4: '╭─────────────────────────────────── Step 0 ───────────────────────────────────╮\n'
  5: '│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ │\n'
  6: '│ ┃ Prompt                         ┃ Completion                     ┃ Reward ┃ │\n'
  7: '│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ │\n'
  8: '│ │ Complete the following Lean 4  │   rw                           │   0.00 │ │\n'
  9: '│ │ code:                          │ [squarefree_iff_nodup_factors] │        │ │\n'
 10: '│ │                                │ at h                           │        │ │\n'
 11: "│ │ ``

In [6]:
# Search for the first occurrence of reward 1.00
with open(input_file, 'r') as f:
    for i, line in enumerate(f):
        if '1.00' in line and '│' in line:
            print(f"Line {i}: {repr(line)}")
            # Show context (5 lines before and after)
            break


Line 30257: '│ │ Complete the following Lean 4  │   simp_all                     │   1.00 │ │\n'


In [7]:
# Also let's check what the separator line looks like exactly
with open(input_file, 'r') as f:
    lines = f.readlines()
    for i, line in enumerate(lines):
        if '├' in line and 'Prompt' not in line:
            print(f"Separator at line {i}: {repr(line)}")
            break

Separator at line 41: '│ ├────────────────────────────────┼────────────────────────────────┼────────┤ │\n'


# start

In [8]:
with open(input_file, 'r') as f:
    lines = f.readlines()

In [9]:
# Find the index of the first 1.00 reward line
start_idx = None
for i, line in enumerate(lines):
    if '│   1.00 │ │' in line:
        start_idx = i
        print(f"Found 1.00 at line {i}: {repr(line[:80])}...")
        break

Found 1.00 at line 30257: '│ │ Complete the following Lean 4  │   simp_all                     │   1.00 │ │'...


In [10]:
# Now find where this block starts (look back for "Complete the following")
block_start = start_idx
for i in range(start_idx, max(0, start_idx-50), -1):
    if 'Complete the following Lean 4' in lines[i]:
        block_start = i
        print(f"Block starts at line {i}")
        break

Block starts at line 30257


In [11]:
# Find where this block ends (look forward for separator ├──────)
block_end = start_idx
for i in range(start_idx, min(len(lines), start_idx+100)):
    if '│ ├────────────────────────────────' in lines[i]:
        block_end = i
        print(f"Block ends at line {i} (separator)")
        break


Block ends at line 30302 (separator)


In [12]:
# Print the block
print("\n" + "="*80)
print("EXTRACTED BLOCK:")
print("="*80)
for i in range(block_start, block_end):
    print(lines[i], end='')


EXTRACTED BLOCK:
│ │ Complete the following Lean 4  │   simp_all                     │   1.00 │ │
│ │ code:                          │ [commutatorElement_def,        │        │ │
│ │                                │ mul_assoc, mul_left_inj,       │        │ │
│ │ ```lean4                       │ mul_right_inj]                 │        │ │
│ │                                │   <;> rw [← mul_assoc]         │        │ │
│ │                                │   <;> simp_all [mul_assoc,     │        │ │
│ │ import Mathlib                 │ mul_left_inj, mul_right_inj]   │        │ │
│ │ import Aesop                   │   <;> group                    │        │ │
│ │                                │   <;> simp_all [mul_assoc,     │        │ │
│ │                                │ mul_left_inj, mul_right_inj]   │        │ │
│ │                                │   <;> group                    │        │ │
│ │                                │   <;> simp_all [mul_assoc,     │        │ │
│ │ theore

In [13]:
def extract_all_reward_blocks(input_path, output_path, target_reward=1.0):
    """
    Extract all table blocks where reward equals target.
    Processes file line-by-line to handle 50MB+ files.
    """
    reward_pattern = f"│   {target_reward:.2f} │ │"
    separator_pattern = "│ ├────────────────────────────────"
    
    blocks = []
    current_block = []
    in_target_block = False
    
    with open(input_path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            # Check if this is a reward line we want
            if reward_pattern in line and "Complete the following Lean 4" in line:
                # Start of new target block
                if current_block:  # Shouldn't happen but safety check
                    pass
                current_block = [line]
                in_target_block = True
                
            elif in_target_block:
                # Check if we hit the separator (end of block)
                if separator_pattern in line:
                    # Save the block
                    blocks.append(''.join(current_block))
                    current_block = []
                    in_target_block = False
                else:
                    # Continue adding to current block
                    current_block.append(line)
            
            # Progress update every 100k lines
            if line_num % 100000 == 0:
                print(f"Processed {line_num} lines, found {len(blocks)} blocks with reward {target_reward}")
    
    # Save to output
    with open(output_path, 'w', encoding='utf-8') as out:
        out.write(f"Total blocks with reward {target_reward}: {len(blocks)}\n")
        out.write("="*80 + "\n\n")
        
        for i, block in enumerate(blocks, 1):
            out.write(f"BLOCK {i}\n")
            out.write("-"*40 + "\n")
            out.write(block)
            out.write("\n" + "="*80 + "\n\n")
    
    return len(blocks)


In [14]:
output_file = "/home/lyz/repos/open-r1-lean/past_runs/run_3_algebra/passed.log"
# Run it
print("Extracting all blocks with reward 1.00...")
count = extract_all_reward_blocks(input_file, output_file, target_reward=1.0)
print(f"\nDone! Extracted {count} blocks.")
print(f"Saved to: {output_file}")

Extracting all blocks with reward 1.00...
Processed 100000 lines, found 8 blocks with reward 1.0
Processed 200000 lines, found 49 blocks with reward 1.0
Processed 300000 lines, found 95 blocks with reward 1.0
Processed 400000 lines, found 124 blocks with reward 1.0
Processed 500000 lines, found 213 blocks with reward 1.0

Done! Extracted 213 blocks.
Saved to: /home/lyz/repos/open-r1-lean/past_runs/run_3_algebra/passed.log
